# Design Choices
- EDA shows that each **component failure** has its own matching **error** and **telemetry sensor**, so my Candidate Precursors are built and tested one **component** at a time using those detectors.

- Starting with comp1, I will test error-based detectors, telemetry-based detectors, and combinations of both to identify the most effective approaches. I will then apply the strongest approaches to comp2, comp3, and comp4 using their respective error and sensor signals

**H and W (warning window)**
- **H = 12 hours, W = 36 hours**, so the warning window covers **48 to 12 hours before each failure**.
- The EDA showed errors appear about 24 hours before a failure and abnormal sensor readings begin about 48 hours before, so this window covers both.
- H is under 24 hours so the error is still inside the window, while still giving 12 hours to act on an alert.
- Failures on a machine are usually at least 15 days apart, so warning windows don't overlap.

**Normal operating time**
- Normal time is every hour that is **not within 48 hours before any failure**.
- We dont consider any recovery period after repairs or maintenance, since the EDA showed sensors go straight back to normal after a component is replaced.
- EDA shows that sensors are consistently running throughout the year indicating no downtime.

**Metrics**
- Lift: how much more often it fires in the W window before events than during normal
operation (a lift of 5 = fires 5x more often).
- Precision: of the alerts it raises (each time it switches from off to on), how often an event
follows within H + W.
- Coverage: how many events it fired at least once before (within their W window), out of all
events.


# Imports

In [1]:
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt

/Users/jaredpaul/Coding/Predictive-Maintenance-RCA/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/jaredpaul/Coding/Predictive-Maintenance-RCA/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Global Variables

In [2]:
EVENT_COMP = "comp1"
H = 12   # hours of warning before the failure
W = 36   # length of the warning window in hours


# Helper Functions

In [3]:
def calc_lift(alert_col):
    rate_in_window = grid_df.loc[grid_df["in_window"], alert_col].mean()
    rate_in_normal = grid_df.loc[grid_df["normal"], alert_col].mean()
    return rate_in_window / rate_in_normal

In [4]:
def calc_precision(alert_col):
    was_on_last_hour = grid_df.groupby("machineID")[alert_col].shift(1, fill_value=False)
    alert_starts_df = grid_df[grid_df[alert_col] & ~was_on_last_hour]
    return alert_starts_df["in_window"].mean()

In [5]:
def calc_coverage(alert_col):
    # Failures whose warning window had at least one alert
    caught = grid_df[grid_df[alert_col] & grid_df["in_window"]]["window_id"].nunique()
    return caught / len(events_df)


# Dataframes

In [6]:
path = kagglehub.dataset_download("arnabbiswas1/microsoft-azure-predictive-maintenance")

telemetry_df = pd.read_csv(f"{path}/PdM_telemetry.csv", parse_dates=["datetime"])
errors_df = pd.read_csv(f"{path}/PdM_errors.csv", parse_dates=["datetime"])
maintenance_df = pd.read_csv(f"{path}/PdM_maint.csv", parse_dates=["datetime"])
failures_df = pd.read_csv(f"{path}/PdM_failures.csv", parse_dates=["datetime"])
machines_df = pd.read_csv(f"{path}/PdM_machines.csv")


<b>Skip failures in the first 48h of data, since there's no history to look back on

In [7]:
data_start = telemetry_df["datetime"].min()

events_df = failures_df[
    (failures_df["failure"] == EVENT_COMP) &
    (failures_df["datetime"] >= data_start + pd.Timedelta(hours=48))
]

print(f"{len(events_df)} {EVENT_COMP} failures")
events_df.head()


181 comp1 failures


,datetime,machineID,failure
1,2015-03-06 06:00:00,1,comp1
7,2015-03-19 06:00:00,2,comp1
12,2015-02-06 06:00:00,3,comp1
17,2015-02-16 06:00:00,4,comp1
20,2015-08-30 06:00:00,4,comp1


In [8]:
grid_df = telemetry_df[["machineID", "datetime"]].copy()

In [9]:
#grid_df

<b> Mark the warning window (48h to 12h before each comp1 failure). <br>
Tag each hour in a window with its failure's ID (`window_id`), so coverage can count how many failures were caught.

In [10]:
grid_df["window_id"] = None

for failure_id, failure in events_df.iterrows():
    window_start = failure["datetime"] - pd.Timedelta(hours=H + W)
    window_end = failure["datetime"] - pd.Timedelta(hours=H)

    grid_df.loc[
        (grid_df["machineID"] == failure["machineID"]) &
        (grid_df["datetime"] >= window_start) &
        (grid_df["datetime"] < window_end),
        "window_id"
    ] = failure_id

grid_df["in_window"] = grid_df["window_id"].notna()


In [11]:
#grid_df[grid_df['in_window'] == True]

<b> Mark normal operating time: every hour that is not within 48h before a failure of any component.

In [12]:
grid_df["normal"] = True

for _, failure in failures_df.iterrows():
    grid_df.loc[
        (grid_df["machineID"] == failure["machineID"]) &
        (grid_df["datetime"] >= failure["datetime"] - pd.Timedelta(hours=H + W)) &
        (grid_df["datetime"] <= failure["datetime"]),
        "normal"
    ] = False


print(f"Total hours:          {len(grid_df)}")
print(f"Warning window hours: {grid_df['in_window'].sum()}")
print(f"Normal hours:         {grid_df['normal'].sum()}")


Total hours:          876100
Warning window hours: 6516
Normal hours:         841422


In [13]:
# grid_df

### Candidate Precursor 1: Error1 Alert


**Story:** component 1 starts to malfunction before it completely fails, and the machine logs an error1 warning as it detects the problem. The EDA showed error1 appears about 24 hours before almost every comp1 failure.<br>

**Detector:** alert whenever error1 is logged on the machine.


In [14]:
DETECTOR_ERROR = "error1"   # comp1's matching error

alerts_df = errors_df[errors_df["errorID"] == DETECTOR_ERROR][["machineID", "datetime"]]
print(f"{len(alerts_df)} {DETECTOR_ERROR} alerts")

1010 error1 alerts


In [15]:
error1_df = errors_df[errors_df["errorID"] == "error1"][["machineID", "datetime"]].copy()
error1_df["error1_alert"] = True
error1_df

,machineID,datetime,error1_alert
0,1,2015-01-03 07:00:00,True
6,1,2015-01-27 04:00:00,True
8,1,2015-03-05 06:00:00,True
9,1,2015-03-20 18:00:00,True
11,1,2015-03-31 23:00:00,True
...,...,...,...
3908,100,2015-10-17 12:00:00,True
3909,100,2015-10-24 23:00:00,True
3912,100,2015-11-07 17:00:00,True
3913,100,2015-11-12 01:00:00,True


In [16]:
#grid_df

<b> Add the error1 alert to the grid: True in any hour where error1 was logged on that machine, False otherwise


In [17]:
grid_df = grid_df.merge(error1_df, on=["machineID", "datetime"], how="left")
grid_df["error1_alert"] = grid_df["error1_alert"].fillna(False).astype(bool)

/var/folders/h3/58sb5ybx6fzgy8l9n1029qd80000gn/T/ipykernel_29615/1630985390.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  grid_df["error1_alert"] = grid_df["error1_alert"].fillna(False).astype(bool)


In [18]:
grid_df[(grid_df['in_window'] == True) & (grid_df['error1_alert'] == True)]

,machineID,datetime,window_id,in_window,normal,error1_alert
1512,1,2015-03-05 06:00:00,1,True,False,True
10585,2,2015-03-18 06:00:00,7,True,False,True
18362,3,2015-02-05 06:00:00,12,True,False,True
27363,4,2015-02-15 06:00:00,17,True,False,True
32043,4,2015-08-29 06:00:00,20,True,False,True
...,...,...,...,...,...,...
859322,99,2015-02-01 06:00:00,741,True,False,True
861838,99,2015-05-17 02:00:00,748,True,False,True
861842,99,2015-05-17 06:00:00,748,True,False,True
868323,100,2015-02-11 06:00:00,758,True,False,True


<b>Metric Results

In [19]:
print(f"Lift of:  {calc_lift('error1_alert'):.1f}")
print(f"Precision: {calc_precision('error1_alert'):.0%}")
print(f"Coverage: {calc_coverage('error1_alert'):.0%}")


Lift of:  30.4
Precision: 19%
Coverage: 100%


### Candidate Precursor 2: sustained high volt
**Story:** as component 1 wears out, the machine draws higher voltage to keep running. The EDA showed volt jumps from ~170 to ~190 about 48 hours before a comp1 failure and stays there.

**Detector:** the 12-hour rolling average of volt is above its 99th percentile of normal operation. Once on, the alert stays on until volt has been below the threshold for 12 hours (to stop it flickering on and off triggering too many alerts for the same problem)


In [20]:
grid_df = grid_df.merge(telemetry_df[["machineID", "datetime", "volt"]], on=["machineID", "datetime"], how="left")

grid_df["volt_12h"] = grid_df.groupby("machineID")["volt"].transform(lambda x: x.rolling(12).mean())

In [21]:
volt_threshold = grid_df.loc[grid_df["normal"], "volt_12h"].quantile(0.99)
print(f"Volt threshold (99th percentile of normal): {volt_threshold:.1f}")

grid_df["volt_alert"] = grid_df["volt_12h"] > volt_threshold


Volt threshold (99th percentile of normal): 191.0


In [22]:
#grid_df[grid_df['volt_12h'].notna()]

In [23]:
grid_df["volt_alert_steady"] = (
    grid_df.groupby("machineID")["volt_alert"]
    .transform(lambda x: x.rolling(12, min_periods=1).max())
    .astype(bool)
)

In [24]:
#grid_df

<b> Metric Results

In [25]:
print(f"Lift:      {calc_lift('volt_alert_steady'):.1f}")
print(f"Precision: {calc_precision('volt_alert_steady'):.0%}")
print(f"Coverage:  {calc_coverage('volt_alert_steady'):.0%}")


Lift:      23.3
Precision: 21%
Coverage:  91%


### Candidate 3: error1 + sustained high volt
**Story:** error1 and high volt are each noisy on their own, but a real comp1 problem should show up in both at the same time.

**Detector:** alert when error1 is logged while the sustained high volt alert (Candidate 2) is on.


In [26]:
grid_df["combined_alert"] = grid_df["error1_alert"] & grid_df["volt_alert_steady"]

<b> metric results

In [27]:
print(f"Lift:      {calc_lift('combined_alert'):.1f}")
print(f"Precision: {calc_precision('combined_alert'):.0%}")
print(f"Coverage:  {calc_coverage('combined_alert'):.0%}")

Lift:      1208.3
Precision: 87%
Coverage:  71%


### Candidate 4: consecutive raw volt outliers
**Story:** if component 1 is failing, raw voltage readings should be extreme several hours in a row, not just once. Testing the potential precursor example from assignment.

**Detector:** raw volt above its 99th percentile of normal operation for 3+ consecutive hours.


In [28]:
raw_volt_threshold = grid_df.loc[grid_df["normal"], "volt"].quantile(0.99)
print(f"Raw volt threshold (99th percentile of normal): {raw_volt_threshold:.1f}")

grid_df["volt_outlier"] = grid_df["volt"] > raw_volt_threshold

Raw volt threshold (99th percentile of normal): 207.2


In [29]:
# How many of the last 3 hours were outliers (per machine)
outliers_in_last_3 = grid_df.groupby("machineID")["volt_outlier"].transform(lambda x: x.rolling(3).sum())

# Alert only if all of the last 3 hours were outliers
grid_df["volt_3_in_a_row"] = outliers_in_last_3 == 3


<b> Metric Results

In [30]:
print(f"Lift:      {calc_lift('volt_3_in_a_row'):.1f}")
print(f"Precision: {calc_precision('volt_3_in_a_row'):.0%}")
print(f"Coverage:  {calc_coverage('volt_3_in_a_row'):.0%}")


Lift:      130.9
Precision: 41%
Coverage:  14%


### Comp1 Candidate Results

| Candidate | Alerts | Lift | Precision | Coverage |
|---|---|---|---|---|
| 1. error1 alert | 1,010 | 30.4 | 19% | 100% |
| 2. Sustained high volt | 867 | 23.3 | 21% | 91% |
| 3. error1 + sustained high volt | 150 | ~1,200 | 87% | 71% |
| 4. 3+ raw volt outliers in a row | 101 | ~131 | 41% | 14% |

**Discussion:**
- **error1** catches every comp1 failure, but only about 1 in 5 alerts is followed by a failure.
- **Sustained high volt** catches most failures and gives an earlier warning (about 48 hours), but has the same false alarm problem.
- **Combining error1 and volt** gives by far the best precision (87%). error1 and high volt each go off on their own during normal operation, but they rarely happen together unless comp1 is actually failing. Requiring both cuts out most of the false alarms. The trade-off is that it misses ~29% of failures.

- **Consecutive raw volt outliers** is rejected. Hourly volt is too noisy for several extreme readings in a row, so it only catches 14% of failures.

<b> I've decided to carry forward the design logic of precursor 1,2 and 3 to the other components for testing but ultimately I believe precursor 3 design will be the best. <br>
(continued in 03_final_precursor_analysis_and_findings.ipynb)
